# TP 1 — Préparer un corpus et entraîner un modèle bigramme

**Travail réalisé par : Emine OULD AGATT**

Ce premier TP présente les étapes qui permettent de transformer un texte brut en séquences utilisables pour l'apprentissage d'un modèle de langage simple, fondé sur les bigrammes.

**Livrable attendu :** un notebook exécuté intégralement, comprenant les exemples de génération avant et après apprentissage ainsi que les réponses argumentées aux questions.


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Charger le corpus et l'explorer

Nous commençons par examiner la taille du document et quelques particularités du texte, notamment les signes de ponctuation.


In [2]:
from urllib.request import urlretrieve
import os

lien = ("https://raw.githubusercontent.com/karpathy/"
       "char-rnn/master/data/tinyshakespeare/input.txt")
urlretrieve(lien, "input.txt")
print("Fichier sauvegardé dans :", os.path.abspath("input.txt"))

with open("input.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

print("Taille du texte:", len(corpus))
print(corpus[:1000])

Fichier sauvegardé dans : /content/input.txt
Taille du texte: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the go

**À analyser :** qu'appelle-t-on un token dans cette expérience ? Quel est l'intérêt de `encoding="utf-8"` ? Les exemples d'entraînement existent-ils déjà dans le fichier ?

**Éléments de réponse :**
- La tokenisation est réalisée caractère par caractère : lettres, espaces, retours à la ligne, etc. Le vocabulaire comprend 65 symboles distincts.
- L'encodage UTF-8 garantit une lecture cohérente du contenu, y compris sur des systèmes dont l'encodage par défaut diffère.
- Le corpus n'est pas fourni sous forme d'exemples indépendants. Il s'agit d'une longue séquence, de plus d'un million de caractères, que nous devons ensuite segmenter pour créer les données d'apprentissage.


## Étape 2 — Associer un identifiant numérique à chaque caractère

In [3]:
caracteres_uniques = sorted(list(set(corpus)))
taille_vocab = len(caracteres_uniques)

char_to_id = {ch: i for i, ch in enumerate(caracteres_uniques)}
id_to_char = {i: ch for i, ch in enumerate(caracteres_uniques)}

texte_vers_ids = lambda s: [char_to_id[c] for c in s]
ids_vers_texte = lambda ids: ''.join([ id_to_char[i] for i in ids])

test_msg = "hii there"
print(texte_vers_ids(test_msg))
print(ids_vers_texte(texte_vers_ids(test_msg)))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


**À analyser :** pourquoi établir un ordre fixe avec `chars` ? Que vérifie l'égalité `decode(encode(message)) == message` ?

**Éléments de réponse :**
- Le tri fournit une correspondance stable entre les caractères et leurs indices. Sans cette opération, l'ordre issu de `set()` pourrait varier entre les exécutions, rendant les encodages incompatibles.
- Le test de reconstruction confirme que l'encodage et le décodage sont réciproques : tous les caractères de la chaîne d'origine sont récupérés sans modification.


## Étape 3 — Constituer les séquences d'entrée et les cibles

Nous réservons 90 % des données à l'entraînement et les 10 % restants à la validation.

In [4]:
t_test = [1,2,3,4,5,6,7,8,9]
taille_bloc = 8
x_ex = t_test[:taille_bloc]
y_ex = t_test[1:taille_bloc + 1]
print("X:", x_ex)
print("Y:", y_ex)

X: [1, 2, 3, 4, 5, 6, 7, 8]
Y: [2, 3, 4, 5, 6, 7, 8, 9]


In [5]:
donnees_tensor = torch.tensor(texte_vers_ids(corpus), dtype=torch.long)

limite = int(0.9 * len(donnees_tensor))
set_entrainement = donnees_tensor[:limite]
set_validation = donnees_tensor[limite:]

taille_bloc = 8
seq_x = set_entrainement[:taille_bloc]
seq_y = set_entrainement[1:taille_bloc + 1]

for t in range(taille_bloc):
    contexte_actuel = seq_x[:t + 1]
    cible = seq_y[t]
    print(contexte_actuel, "->", cible)


tensor([18]) -> tensor(47)
tensor([18, 47]) -> tensor(56)
tensor([18, 47, 56]) -> tensor(57)
tensor([18, 47, 56, 57]) -> tensor(58)
tensor([18, 47, 56, 57, 58]) -> tensor(1)
tensor([18, 47, 56, 57, 58,  1]) -> tensor(15)
tensor([18, 47, 56, 57, 58,  1, 15]) -> tensor(47)
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> tensor(58)


**Points à vérifier :**
1. Quel est le nombre de prédictions associé à un bloc ?
2. Pourquoi séparer les données d'entraînement et celles de validation ?
3. Au temps `t = 3`, quelle séquence constitue le contexte et quel nombre faut-il prévoir ?


**Éléments de réponse :**
1. Une taille de bloc égale à 8 conduit à huit positions pour lesquelles le modèle doit prédire le caractère suivant.
2. Conserver un jeu de validation distinct permet de mesurer le comportement du modèle sur une portion de texte non utilisée pour ajuster ses poids et de repérer un éventuel surapprentissage.
3. Pour `t = 3`, l'entrée correspond aux identifiants `[18, 47, 56, 57]` et l'identifiant cible est `58`.


## Étape 4 — Tirer un mini-lot de séquences

In [6]:
batch_size = 4
taille_bloc = 8

def recup_batch(split):
    donnees = set_entrainement if split == "train" else set_validation
    # On tire des indices de départ au hasard
    indices_debut = torch.randint(len(donnees) - taille_bloc, (batch_size,))

    xb = torch.stack([donnees[i : i + taille_bloc] for i in indices_debut])
    yb = torch.stack([donnees[i + 1 : i + taille_bloc + 1] for i in indices_debut])

    return xb, yb

x_batch, y_batch = recup_batch("train")
print(x_batch.shape, y_batch.shape)

torch.Size([4, 8]) torch.Size([4, 8])


**Questions de compréhension :**
1. Pourquoi limiter les indices de départ à la longueur des données moins `block_size` ?
2. Quels seraient les effets d'un batch de taille `B = 32` ?
3. Qu'apporterait un contexte de longueur `T = 64` ?


**Éléments de réponse :**
1. Cette marge évite de dépasser la fin du corpus lors de la récupération d'une séquence et de ses caractères cibles.
2. Avec `B = 32`, on traite 32 séquences simultanément au lieu de 4 : la mémoire nécessaire augmente, mais l'estimation du gradient bénéficie d'un plus grand nombre d'exemples.
3. `T = 64` allonge le contexte disponible. Un modèle strictement bigramme n'exploite toutefois que le caractère courant ; cette extension devient surtout intéressante pour les Transformers, qui peuvent exploiter les positions antérieures.


## Étape 5 — Obtenir les logits avec une table de correspondance

In [7]:
class ModeleBigramme(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Une table de taille (vocabulaire x vocabulaire)
        self.table_embeddings = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, cibles=None):
        logits = self.table_embeddings(idx)

        if cibles is None:
            loss = None
        else:
            b, t, c = logits.shape
            logits = logits.view(b * t, c)
            cibles = cibles.view(b * t)
            loss = F.cross_entropy(logits, cibles)

        return logits, loss

**À expliquer :** pourquoi les dimensions sont-elles `C × C` ? Comment `d_emb` et `lm_head` interviendront-ils dans le TP 4 ?

**Éléments de réponse :**
- Les `C` lignes représentent les caractères observés et les `C` colonnes les scores associés aux caractères suivants. Avec un vocabulaire de 65 symboles, la table a donc la taille `65 × 65`. Ces valeurs sont des logits, avant normalisation en probabilités.
- Dans le TP 4, les caractères passent d'abord par un embedding de dimension `d_emb = 64`. Après la contextualisation, la projection linéaire `lm_head` produit un score pour chacune des 65 classes du vocabulaire.


## Étapes 6 et 7 — Produire du texte, puis ajuster les poids

### Étape 6 — Ajouter la procédure de génération

In [8]:
def fonction_generer(self, idx, max_nouveaux_tokens):
    for _ in range(max_nouveaux_tokens):
        logits, _ = self(idx)
        # on s'intéresse juste au dernier pas de temps
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        prochain_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, prochain_idx), dim=1)
    return idx

ModeleBigramme.generate = fonction_generer

In [9]:
# Génération avec des poids initiaux (complètement au hasard)
mon_modele = ModeleBigramme(taille_vocab)
start_idx = torch.zeros((1, 1), dtype=torch.long)
print("Avant entrainement:\n")
print(ids_vers_texte(mon_modele.generate(start_idx, max_nouveaux_tokens=100)[0].tolist()))

Avant entrainement:


;&BcvidMXG:DcJeW-',M'v mOfPnTMJeLRXYauWjYbIwmPO?
FYS.WNa&CwHxr-dYK-pn$Nd?Ny.CFbPMlOPcUxciVbvj.z
zRSq


### Étape 7 — Optimiser le modèle

In [10]:
optimiseur = torch.optim.AdamW(mon_modele.parameters(), lr=1e-3)

for etape in range(1000):
    batch_x, batch_y = recup_batch("train")
    predictions, perte = mon_modele(batch_x, batch_y)

    optimiseur.zero_grad(set_to_none=True)
    perte.backward()
    optimiseur.step()

print(f"Loss à la fin : {perte.item():.4f}")

Loss à la fin : 3.8984


In [11]:
# Génération après l'entraînement
print("Après entrainement:\n")
print(ids_vers_texte(mon_modele.generate(start_idx, max_nouveaux_tokens=100)[0].tolist()))

Après entrainement:


f mj rDOEtHro rF3MwXgqsc!r.WNT;DNY'UxcMT!s$
IIg,RGfxBnTHWs!
;'X
KY3qybhX ,JoJaMzYtEVBB-vle.ed
BnxNyg


**Synthèse à rédiger — TP 1**
Pourquoi utilise-t-on uniquement la dernière position temporelle pour générer le caractère suivant ? Pour quelle raison regroupe-t-on les dimensions `B × T` pour calculer la perte ? Mettre enfin en parallèle la perte et l'aspect des textes obtenus avant et après entraînement.


**Synthèse et observations :**
- Pour générer un nouveau symbole, seule la distribution calculée à la dernière position du contexte est nécessaire. Les autres positions décrivent des caractères déjà présents dans la séquence.
- La fonction `cross_entropy` attend des prédictions organisées par observation et par classe. L'aplatissement des dimensions `B` et `T` permet donc de considérer chaque position du mini-lot comme un exemple de classification.
- À l'initialisation, les poids sont aléatoires. Pour 65 possibilités équiprobables, la perte attendue est proche de `−ln(1/65) ≈ 4,17`.
- Sans apprentissage, la génération produit essentiellement une succession incohérente de caractères.
- Après 1 000 mises à jour, la perte diminue légèrement, vers `3,8–3,9`. Le texte reste peu structuré, mais certaines associations de lettres et certains espaces deviennent plus plausibles. Cette amélioration limitée est cohérente avec les capacités réduites d'un modèle bigramme et la durée de l'entraînement.
